# Notebook 4: Long-Term Memory — SEMANTIC (Facts & Preferences)

### The limitation that motivates everything today
Run Notebooks 1–3 again with a **new** `thread_id`: the agent is a stranger again.
Short-term memory belongs to **one conversation**. Real apps need to remember
**the user** — across every conversation, forever.

### The fix: a second component — the **Store**
| | Checkpointer (NB1–3) | Store (NB4–6) |
|---|---|---|
| Scope | ONE thread | Shared ACROSS all threads |
| Lifetime | one conversation | as long as you want |
| Holds | raw message state | clean, structured knowledge |
| Analogy | memory during a phone call | a notebook you keep |

### SEMANTIC memory = *things the agent knows*
Facts and preferences: *"Rahul prefers dark mode." "Priya's company uses Postgres."*

### Human analogy
A **profile card** the teacher keeps for each student: name, level, weak topics.
New class, same card.

### The Store has only 3 operations
```python
store.put(namespace, key, value)     # write
store.get(namespace, key)            # read one
store.search(namespace)              # read many
```
`namespace` = a folder. Convention: scope by user → `("users", user_id, "facts")`.

### Where to use it
Personalization, user profiles, CRM bots, "remember my preferences" features.

### What it connects with later
The *same store* will hold **experiences** (Notebook 5) and **rules** (Notebook 6).
Learn the store once, reuse it three times.

## Step 1 — Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
store = InMemoryStore()   # <- the long-term memory component
print("Setup done.")

## Step 2 — The raw Store API, with nothing else in the way

Write three facts about user `"rahul"`, then read them back.
Notice the namespace: `("users", "rahul", "facts")` — a folder for *this user's facts*.

In [ ]:
store.put(("users", "rahul", "facts"), key="name",     value={"data": "Rahul"})
store.put(("users", "rahul", "facts"), key="fav_food", value={"data": "biryani"})
store.put(("users", "rahul", "facts"), key="goal",
          value={"data": "become a LangChain trainer"})

print("What's in the store for rahul:")
for item in store.search(("users", "rahul", "facts")):
    print(f"  {item.key}: {item.value['data']}")

## Step 3 — An agent that reads its memories every turn

The node does one extra thing compared to Notebook 1:
**before calling the LLM, it loads this user's facts from the store** and puts them
into the system prompt. That's it — long-term memory is "read the store, inject, call LLM".

Note the compile line: **checkpointer** (short-term, Notebook 1) + **store** (long-term, new).
Both together is the standard real-world setup.

In [ ]:
def chatbot(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]

    # ★ Load long-term memories — works no matter which thread we're in
    memories = store.search(("users", user_id, "facts"))
    memory_text = "
".join(f"- {m.key}: {m.value['data']}" for m in memories)         or "No memories yet."

    system = SystemMessage(content=(
        "You are a friendly tutor.
"
        f"Known facts about this user:
{memory_text}"
    ))
    return {"messages": [llm.invoke([system] + state["messages"])]}

builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# ★ BOTH memories: checkpointer (short-term) + store (long-term)
graph = builder.compile(checkpointer=InMemorySaver(), store=store)
print("Agent ready with short-term + long-term memory.")

## Step 4 — The moment that sells it: a new thread, same user

**"Monday"**: Rahul has a normal session (thread `monday`).
**"Friday"**: brand-new thread — but **same `user_id`**. Watch what the agent knows.

In [ ]:
# Session 1 — "Monday"
cfg1 = {"configurable": {"thread_id": "monday", "user_id": "rahul"}}
r = graph.invoke({"messages": [HumanMessage("Let's practice Python loops.")]}, cfg1)
print("Monday AI:", r["messages"][-1].content[:120])

In [ ]:
# Session 2 — "Friday": NEW thread_id, SAME user_id
cfg2 = {"configurable": {"thread_id": "friday", "user_id": "rahul"}}
out = graph.invoke(
    {"messages": [HumanMessage("What do you remember about me?")]}, cfg2
)
print("Friday AI:", out["messages"][-1].content)
# -> It knows name, food, goal — from the STORE, not from this thread's history.

## Step 5 — And a different user is a stranger (as it should be)

In [ ]:
cfg3 = {"configurable": {"thread_id": "friday", "user_id": "priya"}}  # different user!
out = graph.invoke(
    {"messages": [HumanMessage("What do you remember about me?")]}, cfg3
)
print("Priya's AI:", out["messages"][-1].content)
# -> No memories. The store is scoped per user via the namespace.

## Try it yourself

1. Add a new fact: `store.put(("users", "rahul", "facts"), key="city", value={"data": "Hyderabad"})`
   then ask the Friday thread *"where should I visit on the weekend?"* — does it use the city?
2. The one thing to internalize: **`thread_id` changed, `user_id` didn't.**
   Checkpointer is keyed by *which conversation*; the Store is keyed by *who*.
3. In a real app, an LLM step extracts facts from the conversation and calls `store.put`
   automatically — the **langmem** library does this for you
   (`create_memory_store_manager`). One slide is enough for today; the mechanism
   you just built by hand is what's inside it.

**Next notebook:** facts are *what the agent knows*. But what about *what the agent has
**done*** — and what worked? → Notebook 5: Episodic memory.